# Task 3: Association Rules -- Taxonomy-Based Grouping

This notebook describes an approach for grouping products into higher-level categories to improve association rule mining. We focus on **taxonomy-aware association rule mining**, which extends the classical APRIORI algorithm to operate on hierarchical item categories rather than individual items.

## 1. Introduction: The Problem of Item Granularity

The standard **APRIORI algorithm** discovers frequent itemsets and association rules from transactional data by counting co-occurrences of individual items. However, in many real-world retail datasets, items are extremely granular -- for example, a supermarket may carry dozens of pizza variants (Pizza Margherita, Pizza Quattro Formaggio, Pizza Diavola, etc.), each with low individual support.

This granularity creates a fundamental tension: while specific items are what customers actually buy, many purchasing patterns exist at a **conceptual level**. A customer who buys any pizza is likely to also buy beer -- but this rule may never be discovered if each individual pizza variant has support below the minimum threshold. The challenge is to group related items into higher-level categories in a way that reveals these hidden patterns without losing too much specificity.

**Taxonomy-aware association rule mining** addresses this by organizing items into a hierarchy (taxonomy) and mining rules at multiple levels of abstraction. This approach was first formalized by Srikant and Agrawal (1995) in their seminal paper on mining generalized association rules.

## 2. Approach: Taxonomy-Aware Association Rule Mining

### The Concept of Item Taxonomies

An **item taxonomy** is a hierarchical tree structure where leaf nodes represent specific items and internal nodes represent progressively broader categories. For example:

```
Food & Beverage
├── Pizza
│   ├── Pizza Margherita
│   ├── Pizza Quattro Formaggio
│   └── Pizza Diavola
├── Pasta
│   ├── Spaghetti Bolognese
│   └── Penne Arrabiata
└── Beverages
    ├── Cola
    ├── Beer
    └── Orange Juice
```

The key insight is that a transaction containing "Pizza Margherita" implicitly also contains "Pizza" and "Food & Beverage" at higher levels. By encoding transactions at multiple levels, we can discover rules that would be invisible at the item level alone.

### Illustrative Example

Consider a dataset with 1,000 transactions where:
- Pizza Margherita appears in 30 transactions (support = 3%)
- Pizza Quattro Formaggio appears in 25 transactions (support = 2.5%)
- Pizza Diavola appears in 20 transactions (support = 2%)
- Beer appears in 200 transactions (support = 20%)

With a minimum support threshold of 5%, no individual pizza variant qualifies for frequent itemset generation. However, the grouped category **"Pizza"** has support = 7.5% (assuming no overlap), which exceeds the threshold. This enables the discovery of the rule **{Pizza} -> {Beer}**, which captures a genuine and actionable purchasing pattern.

## 3. Algorithm Description

The taxonomy-aware association rule mining algorithm extends APRIORI through the following steps:

### Step 1: Define the Taxonomy

The item taxonomy can be constructed in two ways. **Manual definition** relies on domain experts who assign items to categories based on business knowledge -- this is common in retail where product hierarchies already exist (e.g., department > category > subcategory > SKU). **Automatic construction** uses clustering or text similarity on item names/descriptions to build the hierarchy algorithmically, which is useful when no pre-existing categorization exists.

### Step 2: Encode Transactions at Multiple Levels

Each transaction is augmented with ancestor items from the taxonomy. If a customer bought "Pizza Margherita" and "Beer", the encoded transaction becomes: {Pizza Margherita, Pizza, Food & Beverage, Beer, Beverages, Food & Beverage}. Duplicate ancestor entries are removed, so each category appears at most once per transaction. This step increases the effective support of higher-level categories.

### Step 3: Run APRIORI at Each Level

The standard APRIORI algorithm is applied to the augmented transactions. Because higher-level categories now appear in more transactions, they are more likely to meet the minimum support threshold. The algorithm naturally discovers rules at all levels of the taxonomy -- from specific item-level rules (if support is sufficient) to broad category-level rules.

### Step 4: Filter Redundant Rules

A critical post-processing step removes **redundant generalized rules**. A rule at a higher level is considered redundant if it is merely a generalization of a more specific rule with similar support and confidence. For example, if both {Pizza Margherita} -> {Beer} and {Pizza} -> {Beer} exist with comparable metrics, the more specific rule is preferred because it is more actionable. However, if only the generalized rule meets the support threshold, it is retained as it provides information not available at the specific level.

### Cross-Level Rules

The algorithm can also discover **cross-level rules** that mix abstraction levels, such as {Pizza} -> {Cola} (category to item). These rules arise when a broad category on one side and a specific item on the other both have sufficient support. Cross-level rules can be particularly insightful but require careful interpretation to avoid misleading conclusions.

## 4. Advantages of Taxonomy-Based Grouping

Taxonomy-aware association rule mining offers several significant benefits over standard item-level mining:

- **Discovers hidden patterns**: Items with individually low support become visible when grouped with related items. This is particularly important for the **long-tail problem** in retail -- the majority of items are purchased infrequently, but collectively they represent a large share of transactions. Grouping rare items with their popular siblings enables pattern discovery that would otherwise be impossible.

- **Reduces rule volume**: Mining at the item level on a large product catalog (10,000+ SKUs) can generate millions of rules, most of which are noise. Taxonomy-based grouping dramatically reduces the number of rules by consolidating similar items, producing a more manageable and interpretable set of patterns.

- **Captures conceptual relationships**: From a business perspective, all pizza variants often behave similarly in terms of purchasing context. The rule {Pizza} -> {Beer} captures this conceptual relationship cleanly, without requiring separate rules for each pizza variant.

- **Aligns with business decision-making**: Marketing campaigns, store layout decisions, and inventory management typically operate at the category level rather than the individual SKU level. Category-level rules are directly actionable for these use cases -- a store manager can place the beer aisle near the pizza section based on {Pizza} -> {Beer} without needing to know which specific pizza drives the association.

- **Handles data sparsity**: In datasets with many unique items and relatively few transactions, item-level support counts are often too low to produce meaningful rules. Taxonomy grouping effectively increases the "sample size" for each category, enabling rule discovery even with sparse data.

## 5. Disadvantages of Taxonomy-Based Grouping

Despite its benefits, taxonomy-aware mining has several important limitations:

- **Taxonomy construction is costly or noisy**: Manual taxonomy creation requires domain expertise and is time-consuming, especially for large product catalogs. Automatic clustering-based approaches avoid manual effort but often produce noisy or semantically inconsistent groupings. The quality of discovered rules is directly bounded by the quality of the taxonomy.

- **Loss of specificity**: The rule {Pizza Margherita} -> {Beer} is more actionable than {Pizza} -> {Beer}. For targeted recommendations (e.g., suggesting beer to a customer with Pizza Margherita in their cart), the specific rule is strictly superior. Taxonomy-based grouping trades this specificity for broader coverage, which may not always be the right trade-off.

- **Choosing the right abstraction level is subjective**: Should frozen pizzas and fresh pizzas be in the same category? Should craft beer and mass-market beer be grouped? These decisions are inherently subjective and can significantly affect which rules are discovered. There is no principled way to determine the "optimal" level of abstraction without experimentation.

- **Cross-level rules can be misleading**: Rules that mix abstraction levels (e.g., {Pizza} -> {Heineken}) are difficult to interpret. Does this mean all pizza buyers prefer Heineken specifically, or is it just that Heineken is the most popular beer overall? Without careful analysis, cross-level rules can suggest false specificity.

- **Computational overhead increases**: Encoding transactions at multiple taxonomy levels increases the effective number of items per transaction, which directly increases the computational cost of the APRIORI algorithm. With a deep taxonomy (4+ levels), this overhead can be substantial.

- **Risk of over-generalization**: Grouping unlike items under a common category can produce spurious rules. For example, grouping "frozen pizza" and "restaurant-style fresh pizza" into a single "Pizza" category may obscure fundamentally different purchasing contexts. The association {Frozen Pizza} -> {Microwave Meals} would be lost if frozen pizza is merged with fresh pizza variants.

## 6. Connection to Our Dataset

Our mood prediction dataset contains app usage data across **12 app categories** (e.g., communication, social, entertainment, tools, etc.). These categories already represent a form of taxonomy -- each category groups multiple individual apps into a meaningful higher-level concept.

In **iteration 107** of our pipeline, we tested whether grouping these 12 categories into **4 super-categories** would improve classification performance:
- `social_communication` (merging social and communication apps)
- `entertainment_leisure` (merging entertainment, games, and media apps)
- `productivity_work` (merging productivity, tools, and education apps)
- `miscellaneous` (remaining apps)

The result was a **drop in F1 score of 0.032**, indicating that the grouping lost meaningful signal. This outcome is consistent with the disadvantages described above -- specifically, the original 12 categories already captured meaningful distinctions between app types that our classifier could exploit. Merging them into 4 super-categories constituted over-generalization, collapsing informative variation into overly broad buckets.

This empirical result illustrates a key lesson from taxonomy-based approaches: **grouping is beneficial when individual items are too sparse to analyze meaningfully, but harmful when the original granularity already carries predictive signal**. In association rule mining terms, you should group items when their individual support is too low, but preserve specificity when items have sufficient frequency to stand on their own.

## 7. References

1. Srikant, R., & Agrawal, R. (1995). Mining Generalized Association Rules. *Proceedings of the 21st International Conference on Very Large Data Bases (VLDB)*, 407-419.

2. Agrawal, R., Imielinski, T., & Swami, A. (1993). Mining Association Rules between Sets of Items in Large Databases. *Proceedings of the 1993 ACM SIGMOD International Conference on Management of Data*, 207-216.

3. Agrawal, R., & Srikant, R. (1994). Fast Algorithms for Mining Association Rules. *Proceedings of the 20th International Conference on Very Large Data Bases (VLDB)*, 487-499.

4. Han, J., & Fu, Y. (1995). Discovery of Multiple-Level Association Rules from Large Databases. *Proceedings of the 21st International Conference on Very Large Data Bases (VLDB)*, 420-431.